# An introduction to F# types and type inference

F# checks types before executing a cell. Type inference keeps many annotations optional while retaining static checking. Unlike Python's gradual typing, there is no separate `mypy` extension to enable. The compiler supports generic types, discriminated unions and user-defined types. This notebook covers the same examples as the Python type-annotation introduction; its original filename is retained for comparison.

## Finding errors by type checking
A string cannot be added to an integer. The commented example would be rejected before execution. Converting the text explicitly gives the intended result. The input is a fixed string so **Run All** remains reproducible; replace it with `System.Console.ReadLine()` for an interactive console experiment.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let numberText = "41"
// let invalid = numberText + 1   // compiler error: string and int differ
let number = int numberText
printfn "It is %d." (number + 1)

## Type annotations
Put `: type` after a parameter or a function's parameter list to specify its result type. Function application uses spaces; parentheses group expressions.

In [ ]:
let add (a: int) (b: int) : int = a + b
add 2 3

`add "Hello " "Karl"` is rejected by F# even if never executed. Its inferred function type is `int -> int -> int`. To concatenate strings, define a separate function with the appropriate type.

In [ ]:
let greet name = "Hello " + name
greet "Karl"

The notebook displays inferred signatures when functions are defined. Annotations are part of compile-time checking, rather than a runtime `__annotations__` dictionary. Here F# infers `int -> int -> int` from subtraction.

In [ ]:
let sub a b = a - b
sub 5 2

## Built-in types
F# supports `int`, `float`, `string`, `bool`, homogeneous lists, heterogeneous tuples, and typed maps. A list cannot mix `int` and `float`; explicitly convert its elements when required. `average` rejects the empty list because its arithmetic mean is undefined.

In [ ]:
let average (numbers: float list) : float =
    if List.isEmpty numbers then invalidArg "numbers" "The mean of an empty list is undefined."
    List.sum numbers / float (List.length numbers)
average [1.; 2.; 3.; 4.]

## Custom types
A record is a concise immutable data type. Functions operate on its values. `unit` is the result type for actions such as printing, corresponding to a Python function without a returned value.

In [ ]:
type Person = { Name: string }
let greetPerson (person: Person) = sprintf "Hello, %s!" person.Name
let salve person : unit = printfn "%s" (greetPerson person)
let jc = { Name = "Julius Caesar" }
salve jc

## Union types
The original example accepts either a single name or a given/family name dictionary. A discriminated union describes those two possibilities explicitly, and pattern matching checks every case. A record ensures both structured names are present.

In [ ]:
type FullName = { Given: string; Family: string }
type Name = SimpleName of string | StructuredName of FullName
let greetName name =
    match name with
    | SimpleName text -> "Hi " + text + "!"
    | StructuredName person -> sprintf "Bienvenido, Señor %s %s." person.Given person.Family
greetName (SimpleName "Alice")
greetName (StructuredName { Given = "Esteban"; Family = "Ramirez" })

## Generic functions
F# infers type variables automatically. `swap` has type `'a * 'b -> 'b * 'a`: its tuple elements need not have the same type. No `TypeVar` declarations are needed.

In [ ]:
let swap (x, y) = y, x
swap (1, 2)
swap ('a', 'b')
swap (1, 'a')

Explicit generic annotations are available when they improve clarity. `rotate` has three independent type parameters.

In [ ]:
let rotate<'u, 'v, 'w> (x: 'u, y: 'v, z: 'w) : 'v * 'w * 'u = y, z, x
rotate (1, 'a', true)

## Recursive types
An arithmetic expression is a number, a variable, or an expression built from an operator and two operands. A recursive discriminated union encodes this structure directly. An operator union makes unsupported operator strings impossible.

In [ ]:
type Operator = Add | Subtract | Multiply | Divide
type Expression =
    | Number of float
    | Variable of string
    | Binary of Expression * Operator * Expression

The function `differentiate(e, x)` takes an arithmetic expression `e` and a variable `x`
and compute the derivative of `e` with respect to `x`.

In [ ]:
let rec differentiate expr variable =
    match expr with
    | Number _ -> Number 0.
    | Variable name -> Number (if name = variable then 1. else 0.)
    | Binary (lhs, op, rhs) ->
        let dLhs = differentiate lhs variable
        let dRhs = differentiate rhs variable
        match op with
        | Add -> Binary (dLhs, Add, dRhs)
        | Subtract -> Binary (dLhs, Subtract, dRhs)
        | Multiply -> Binary (Binary (dLhs, Multiply, rhs), Add, Binary (lhs, Multiply, dRhs))
        | Divide ->
            let numerator = Binary (Binary (dLhs, Multiply, rhs), Subtract, Binary (lhs, Multiply, dRhs))
            let denominator = Binary (rhs, Multiply, rhs)
            Binary (numerator, Divide, denominator)

F# checks the recursive branches without suppressions or forward stubs. The product rule uses the original right operand in its first term. The second Python version accidentally used its derivative there; that error is corrected in this translation.

In [ ]:
let polynomial = Binary (Variable "x", Multiply, Variable "x")
differentiate polynomial "x"
let quotient = Binary (Variable "x", Divide, Variable "y")
differentiate quotient "x"
checkNotebook (differentiate (Number 3.) "x" = Number 0.)
checkNotebook (differentiate (Variable "x") "x" = Number 1.)